# Emotion Detection & Topic Modelling

In [ ]:
import isodate
import json
from openai import OpenAI
from tqdm.notebook import tqdm

from openai_wrapper.batch_process import BatchProcessOpenAI
from utils.gcs import list_from_gcs, load_from_gcs
from utils.requests import fetch_with_retries

In [ ]:
from keys import OPENAI_KEY
client = OpenAI(api_key=OPENAI_KEY)

## OpenAI Batch Processing

In [ ]:
tariffs_transcripts = BatchProcessOpenAI(
    key = OPENAI_KEY,
    query = 'US Tariffs',
    filename = 'transcripts.csv'
    )
tariffs_transcripts.process_batch(
    id_field='videoId',
    text_field='transcript'
)

In [ ]:
tariffs_transcripts.retrieve_batch()

In [ ]:
tariffs_transcripts.download_output()

In [ ]:
with open('data/us-tariffs_transcripts.jsonl', 'r', encoding='utf-8') as file:
    filter_ids = [i['custom_id'] for i in [json.loads(line) for line in file if line.strip()]]

In [ ]:
tariffs_commentThreads = BatchProcessOpenAI(
    key = OPENAI_KEY,
    query = 'US Tariffs',
    filename = 'commentThreads.csv'
    )
tariffs_commentThreads.process_batch(
    id_field='id',
    text_field='textOriginal',
    filter_ids=filter_ids,
    filter_col='videoId'
)

In [ ]:
tariffs_commentThreads.retrieve_batch()

In [ ]:
tariffs_commentThreads.download_output()

In [ ]:
tariffs_commentThreadsreplies = BatchProcessOpenAI(
    key = OPENAI_KEY,
    query = 'US Tariffs',
    filename = 'commentThreadsreplies.csv'
    )
tariffs_commentThreadsreplies.process_batch(
    id_field='id',
    text_field='textOriginal',
    filter_ids=filter_ids,
    filter_col='videoId'
)

In [ ]:
tariffs_commentThreadsreplies.retrieve_batch()

In [ ]:
tariffs_commentThreadsreplies.download_output()

## Consolidate Datasets

In [ ]:
def reshape_download(
    query:str,
    filename:str,
    bucket_name:str = 'youtube-us-tariffs2'
) -> dict:
    # Get list of files
    blobnames = list_from_gcs(bucket_name, '-'.join(query.split()).lower(), filename)

    # Relevant cols
    cols = {
        'search.csv': [
            'videoId', 'channelId', 'channelTitle', 'title', 'description',
            'publishTime', 'publishedAt', 'liveBroadcastContent'
            ],
        'videos.csv': [
                'id', 'viewCount', 'favoriteCount', 'likeCount', 'commentCount', 'topicCategories'
            ],
        'commentThreads.csv': [
                'id', 'channelId', 'videoId', 'authorDisplayName', 
                'textDisplay', 'totalReplyCount', 'likeCount', 
                'updatedAt', 'publishedAt'
            ],
        'commentThreadsreplies.csv': [
                'id', 'parentId', 'channelId', 'videoId', 'authorDisplayName',
                'textDisplay', 'likeCount', 
                'updatedAt', 'publishedAt'
            ],
        'transcripts.csv': [
                'videoId', 'language', 'is_generated', 'transcript'
            ]
    }
    id_col = 'id' if filename == 'videos.csv' else 'videoId'
    
    # Download from GCS
    ids = set()
    all_data = []
    for blob in tqdm(blobnames, desc="Downloading blobs", unit="blob", total=len(blobnames)):
        data = load_from_gcs(bucket_name, blob_name=blob)
        if data:
            all_data += [x for x in data if x[id_col] in filter_ids]

    # Clean data
    seen = set()
    cleaned_data = []

    for item in all_data:
        key = item[id_col]
        if key not in seen:
            seen.add(key)
            cleaned_data.append({k: v for k, v in item.items() if k in cols[filename]})

    # Save as JSON
    if cleaned_data:
        with open(f"data/{filename.replace('.csv', '')}.json", 'w') as f:
            json.dump(cleaned_data, f, indent=2)

In [ ]:
query = 'US Tariffs'

reshape_download(query, 'search.csv')
reshape_download(query, 'videos.csv')
reshape_download(query, 'commentThreads.csv')
reshape_download(query, 'commentThreadsreplies.csv')
reshape_download(query, 'transcripts.csv')

In [ ]:
with open('data/us-tariffs_transcripts.jsonl', 'r', encoding='utf-8') as file:
    ids = [i['custom_id'] for i in [json.loads(line) for line in file if line.strip()]]

url = 'https://www.googleapis.com/youtube/v3/videos'
params = {
    'part': 'id,contentDetails',
    'key': OPENAI_KEY
}
all_data = {}
closed_account = []
with tqdm(total=len(filter_ids), desc=f"Retrieving content details") as pbar:
    for i in filter_ids:
        params['id'] = i
        try:
            data = await fetch_with_retries(url, params)
            cleaned_data = data[0]['items'][0]
            all_data[cleaned_data['id']] = isodate.parse_duration(cleaned_data['contentDetails']['duration']).total_seconds()
        except:
            all_data[cleaned_data['id']] = -1

        pbar.update(1)